This notebook walks through using this repo to evaluate model activations from start to end.

# imports

In [1]:
import os
import numpy as np

from snap.wrapper import TorchWrapper
from snap.experiment import Experiment
from snap.nsd_data import get_neural_data
import snap.models as models
from snap.regression_utils_dd import regression_metric, regression

In [2]:
%load_ext autoreload
%autoreload 2

# set important variables

In [3]:
model_name = 'resnet50'
region = 'V1v'
pooling = None
rand_proj_dim = None
reg = None
training = True
num_samples = None
num_voxels = None
random_voxels = False
pca = True
alpha_per_target = False
empirical_only = False
dataset = 'response_demo'
min_ncsnr = 0.2

batch_size = 128
shuffle = False
workers = 4

data_root = '/mnt/home/alargen/SNAP/snap_analysis_data/pca_reg/' # path to save results
nsd_root = '/mnt/ceph/users/alargen/small_nsd/DeepJuiceDev/juicyfruits/nsd_subset' # path to nsd data

In [4]:
if training:
    trained = True
else:
    trained = False

pretrained = {True: 'pretrained',
              False: 'untrained'
              }

loader_kwargs = {'batch_size': batch_size,
                 'shuffle': shuffle,
                 'num_workers': workers,
                 'pin_memory': True,
                }              

device = 'cuda'

# load in model

In [5]:
model_kwargs = {'name': model_name,
                'pretrained': trained,
                'device': device}
model, layers, identifier, img_transforms = models.get_model(**model_kwargs)
model_wrapped = TorchWrapper(model,
                            layers=layers,
                            identifier=identifier,
                            activation_pooling=pooling)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /dev/shm/.cache-alargen/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 145MB/s] 


# load in nsd data

In [6]:
data_loader_neural, images, labels = get_neural_data(region=region, dataset=dataset,
                                                    loader_kwargs=loader_kwargs,
                                                    data_path=nsd_root, num_samples=num_samples,
                                                    image_transforms=img_transforms, subj_subset=[1],
                                                    num_voxels=num_voxels, random_voxels=random_voxels,
                                                    min_ncsnr=min_ncsnr)

Shape of images: torch.Size([1000, 3, 224, 224])
Shape of brain responses: torch.Size([1000, 568])


In [7]:
y = labels['responses']

# pipeline regression

In [50]:
# Create the Experiment Class and pass additional metrics
regression_kwargs = {'num_trials': 5,
                    'reg': reg,
                    'num_points': 5,
                    'with_pca': pca,
                    'alpha_per_target': alpha_per_target,
                    'empirical_only': empirical_only,
                    }

metric_fns = [regression_metric]
exp = Experiment(model_wrapped,
                metric_fns=metric_fns,
                rand_proj_dim=rand_proj_dim)

# Extract the activations of the layers passed above
# using data_loader (only uses the inputs)
exp.get_activations(data_loader_neural)

# Compute metrics
metric_kwargs = {'debug': False,
                'epsilon': 1e-14
                } | regression_kwargs | model_kwargs

exp_metrics = exp.compute_metrics(images=images,
                                    labels=labels,
                                    **metric_kwargs)
layers = exp_metrics['layers']

Getting layer activations...


/mnt/sw/nix/store/29h1dijh98y9ar6n8hxv78v8zz2pqfzf-python-3.11.7-view/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/mnt/sw/nix/store/29h1dijh98y9ar6n8hxv78v8zz2pqfzf-python-3.11.7-view/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Computing metrics...
Computing metrics for ['layer1.0.relu', 'layer1.1.relu', 'layer1.2.relu', 'layer2.0.relu', 'layer2.1.relu', 'layer2.2.relu', 'layer2.3.relu', 'layer3.0.relu', 'layer3.1.relu', 'layer3.2.relu', 'layer3.3.relu', 'layer3.4.relu', 'layer3.5.relu', 'layer4.0.relu', 'layer4.1.relu', 'layer4.2.relu']
{'layer1.0.relu': torch.Size([1000, 802816]), 'layer1.1.relu': torch.Size([1000, 802816]), 'layer1.2.relu': torch.Size([1000, 802816]), 'layer2.0.relu': torch.Size([1000, 401408]), 'layer2.1.relu': torch.Size([1000, 401408]), 'layer2.2.relu': torch.Size([1000, 401408]), 'layer2.3.relu': torch.Size([1000, 401408]), 'layer3.0.relu': torch.Size([1000, 200704]), 'layer3.1.relu': torch.Size([1000, 200704]), 'layer3.2.relu': torch.Size([1000, 200704]), 'layer3.3.relu': torch.Size([1000, 200704]), 'layer3.4.relu': torch.Size([1000, 200704]), 'layer3.5.relu': torch.Size([1000, 200704]), 'layer4.0.relu': torch.Size([1000, 100352]), 'layer4.1.relu': torch.Size([1000, 100352]), 'layer4.

Layer: 100%|██████████| 16/16 [00:32<00:00,  2.02s/it]


Computing regression_metric...


...


 N: 100352, p: 600, Best Alpha: 1e-11, with pca: True, feat_scaler: StandardScaler()


/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=1.91139e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=1.91718e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=1.93851e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=1.97514e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="po


 N: 100352, p: 800, Best Alpha: 10000.0, with pca: True, feat_scaler: StandardScaler()


Layer: 100%|██████████| 16/16 [01:44<00:00,  6.52s/it]

Metric Computation completed!


# "manual" regression

a walkthrough of how `regression_utils_dd.py` does its regression

## get model activations

In [8]:
metric_fns = [regression_metric]
exp = Experiment(model_wrapped,
                metric_fns=metric_fns,
                rand_proj_dim=rand_proj_dim)

acts = exp.get_activations(data_loader_neural)

Getting layer activations...


In [9]:
# choose a single layer to simplify things
layer = 'layer1.0.relu'
feat = acts[layer]

## regression

### imports + variables

In [10]:
import torch
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline

In [11]:
pvals = [600]
cent = True
num_trials = 1
reg = None
alpha_per_target = False
scoring = 'explained_variance'
with_pca = True
scale_feats = True
scale_y = True
n_folds = 5

### setting other vars based on inputs

In [12]:
P, N = feat.shape
C = y.shape[-1]

if pvals is None:
    pvals = [int(.6*P), int(.8*P)]
elif isinstance(pvals, (int, float)):
    pvals = [pvals]

if cent:
    with_mean = True
else:
    with_mean = False

if alpha_per_target:
    err_reg = np.zeros((len(pvals), y.shape[1]))
else:
    err_reg = np.zeros(len(pvals))

errors = {'pvals': pvals,
            'P': P,
            'N': N,
            'C': C,
            'cent': cent,
            'reg': err_reg, 

            'gen_errs': np.zeros((num_trials, len(pvals), C)),
            'tr_errs': np.zeros((num_trials, len(pvals), C)),
            'test_errs': np.zeros((num_trials, len(pvals), C)),

            'r2_gen': np.zeros((num_trials, len(pvals), C)),
            'r2_tr': np.zeros((num_trials, len(pvals), C)),
            'r2_test': np.zeros((num_trials, len(pvals), C)),

            'pearson_tr': np.zeros((num_trials, len(pvals), C)),
            'pearson_test': np.zeros((num_trials, len(pvals), C)),
            'pearson_gen': np.zeros((num_trials, len(pvals), C)),

            'gen_norm': np.zeros((num_trials, len(pvals), C)),
            'tr_norm': np.zeros((num_trials, len(pvals), C)),
            'test_norm': np.zeros((num_trials, len(pvals), C)),
            }

if reg is None:
    alphas = np.logspace(-15, 10, 26).tolist() # do 51 steps to get between too
elif isinstance(reg, (int, float)):
    alphas = [reg]
else:
    alphas = reg


### regression loop

In [14]:
for i, p in enumerate(pvals):
    best_alpha = None
    for j in range(num_trials):

        idx, idx_test = train_test_split(np.arange(0, P, 1), train_size=p, random_state=j)
        assert len(set(idx)) == p
        assert len(set(idx_test)) == P - p

        y_tr = y[idx]
        y_test = y[idx_test]
        feat_tr = feat[idx]

        feat_scaler = StandardScaler(with_mean=(with_mean and scale_feats), 
                                        with_std=scale_feats)
        if best_alpha is None:
            ridge_reg = Ridge()
        else:
            ridge_reg = Ridge(alpha=best_alpha)

        if with_pca: # normally saves the PCAs for faster computations later
            # pca_file_name = f'model_{name}_pretrained_{pretrained}_layer_{layer}_p_{p}_cent_{cent}_scaled_{scale_feats}_trial_{j}'
            # path = f'/mnt/home/alargen/SNAP/snap_analysis_data/pca_decomps/{pca_file_name}'
            # if os.path.isfile(path):
            #     with open(path, "rb") as f:
            #         all_feat = pickle.load(f)
            #         feat_tr = all_feat[idx]
            # else:
            norm_feat_tr = feat_scaler.fit_transform(feat_tr)
            norm_feat = feat_scaler.transform(feat)

            pca = PCA(n_components=p)
            feat_tr = pca.fit_transform(norm_feat_tr)
            all_feat = pca.transform(norm_feat)
                # with open(path, 'wb') as f:
                #     pickle.dump(all_feat, f)

            pipeline = Pipeline([
                ('ridge', ridge_reg)
            ])
        else:
            pipeline = Pipeline([
                ('feat_scaler', feat_scaler),
                ('ridge', ridge_reg)
            ])
            all_feat = feat

        y_scaler = StandardScaler(with_mean=(with_mean and scale_y), 
                                    with_std=scale_y)
        regr = TransformedTargetRegressor(regressor=pipeline,
                                            transformer=y_scaler)

        if best_alpha is None: # need to search for good alpha
            param_grid = {'regressor__ridge__alpha': alphas}
            shuffle = not alpha_per_target # don't want folds to be different when fitting to each voxel
            if shuffle:
                kf = KFold(n_splits=n_folds, shuffle=shuffle, random_state=0)
            else:
                kf = KFold(n_splits=n_folds, shuffle=shuffle)
            gs = GridSearchCV(regr, param_grid, cv=kf, scoring=scoring, n_jobs=-1)

            if alpha_per_target: # pipeline doesn't corretly handle RidgeCV (or generally EstimatorCV)
                best_alpha = torch.zeros(y.shape[1]) # one per voxel
                y_hat = torch.zeros_like(y) # each col is a voxel

                for y_idx in range(y.shape[1]):
                    single_y_tr = y_tr[:, y_idx]
                    
                    gs.fit(np.array(feat_tr), np.array(single_y_tr))
                    best_alpha[y_idx] = gs.best_params_['regressor__ridge__alpha']

                    single_y_hat = torch.from_numpy(gs.predict(np.array(all_feat)))
                    y_hat[:, y_idx] = single_y_hat
                best_alpha = best_alpha.numpy()

                print_alpha = min(best_alpha), max(best_alpha)

            else:
                gs.fit(np.array(feat_tr), np.array(y_tr))
                best_alpha = gs.best_params_['regressor__ridge__alpha']
                
                y_hat = torch.from_numpy(gs.predict(np.array(all_feat)))
                print_alpha = best_alpha

            print(f'\n N: {N}, p: {p}, Best Alpha: {print_alpha}, with pca: {with_pca}, feat_scaler: {feat_scaler}')
            errors['reg'][i] = best_alpha/p

            del gs, kf, param_grid

        else: #have alpha, do regression as normal
            regr.fit(np.array(feat_tr), np.array(y_tr))

            y_hat = torch.from_numpy(regr.predict(np.array(all_feat)))

        y_hat_tr = y_hat[idx]
        y_hat_test = y_hat[idx_test]

        tr_cent = y_tr - y_tr.mean(0, keepdim=True)
        test_cent = y_test - y_test.mean(0, keepdim=True)
        gen_cent = y - y.mean(0, keepdim=True)

        # Compute overall (scalar) normalization factors
        tr_norm = (tr_cent**2).mean(0).sum()
        test_norm = (test_cent**2).mean(0).sum()
        gen_norm = (gen_cent**2).mean(0).sum()

        tr_err = ((y_hat_tr - y_tr)**2).mean(0) / tr_norm
        test_err = ((y_hat_test - y_test)**2).mean(0) / test_norm
        gen_err = ((y_hat - y)**2).mean(0) / gen_norm

        r2_tr = 1 - tr_err
        r2_test = 1 - test_err
        r2_gen = 1 - gen_err

        def pearsonr(pred, target):
            yc = target - target.mean(0, keepdim=True)
            yhatc = pred - pred.mean(0, keepdim=True)
            return (yc*yhatc).sum(0)/torch.sqrt((yc**2).sum(0)*(yhatc**2).sum(0))

        pearson_tr = pearsonr(y_hat_tr, y_tr)
        pearson_test = pearsonr(y_hat_test, y_test)
        pearson_gen = pearsonr(y_hat, y)

        errors['gen_norm'][j, i] = gen_norm.cpu().numpy()
        errors['tr_norm'][j, i] = tr_norm.cpu().numpy()
        errors['test_norm'][j, i] = test_norm.cpu().numpy()

        errors['gen_errs'][j, i] = gen_err.cpu().numpy()
        errors['tr_errs'][j, i] = tr_err.cpu().numpy()
        errors['test_errs'][j, i] = test_err.cpu().numpy()

        errors['r2_gen'][j, i] = r2_gen.cpu().numpy()
        errors['r2_tr'][j, i] = r2_tr.cpu().numpy()
        errors['r2_test'][j, i] = r2_test.cpu().numpy()

        errors['pearson_tr'][j, i] = pearson_tr.cpu().numpy()
        errors['pearson_test'][j, i] = pearson_test.cpu().numpy()
        errors['pearson_gen'][j, i] = pearson_gen.cpu().numpy()

del feat_tr, all_feat #, norm_feat, norm_feat_tr, feat_pca, feat_tr_pca
del y_tr, y_test, y_hat, y_hat_tr, y_hat_test # norm_y, norm_y_tr, norm_y_test,
del regr
# feat, feat_tr, y = 0, 0, 0
torch.cuda.empty_cache()

/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/mnt/home/alargen/python/py_venvs/SNAP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:255: UserWarning: Singular matrix in solving dual


 N: 802816, p: 600, Best Alpha: 100000.0, with pca: True, feat_scaler: StandardScaler()


## theoretical error stuff

In [15]:
from snap.regression_utils_dd import gen_error_theory
from snap.metrics import compute_spectrum

In [16]:
spectrum_dict = compute_spectrum(acts, images, labels)

Layer: 100%|██████████| 16/16 [00:32<00:00,  2.02s/it]


In [17]:
eigs = spectrum_dict['cent'][layer]['eigs']
weights = spectrum_dict['cent'][layer]['weights']['responses']
reg = best_alpha #1e-14 #
theory = gen_error_theory(eigs, weights, reg, pvals)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


E_i sum: 990.5024341651376


In [18]:
theory['E_i'].shape

(1, 1000)